# HW2: Query-based детектор (RT-DETR)
Навчаємо 2 варіанти RT-DETR (baseline і tuned) по 3 епохи, після кожної епохи зберігаємо ваги.
Кожну епоху оцінюємо на 1500 випадкових кадрах train і на повному val (mAP, AP50, AP_small),
так само, як інші моделі, щоб результати можна було порівняти.

In [1]:
!pip -q install -U pycocotools ultralytics

import os
import io
import re
import time
import json
import glob
import random
import contextlib
import numpy as np
import pandas as pd
import cv2

import torch
from torch.utils.data import Dataset, DataLoader, Subset

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR
from ultralytics.data.utils import img2label_paths
from IPython.display import display
from google.colab import drive

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
IMG_SIZE = 1024
GRAY = 114
FRAME_STEP = 3
SEED = 42
BATCH = 8

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Device: {device}')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Device: cuda


In [2]:
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Colab Notebooks/multiple-object-video-detection-hw-2-ucu-ai-2026.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(EXTRACT_DIR):
    print('Розпакування архіву...')
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print('Готово')

ROOT = EXTRACT_DIR
CKPT_DIR = '/content/drive/MyDrive/visdrone_ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
print('ROOT:', ROOT, '| CKPT_DIR:', CKPT_DIR)

Mounted at /content/drive
Розпакування архіву...
Готово
ROOT: /content/dataset | CKPT_DIR: /content/drive/MyDrive/visdrone_ckpt


Читаємо COCO-анотації в таблиці. `ioa_boxes` рахує, яка частка об'єкта лежить в ignored-зоні (`iscrowd=1`).
`VisDroneEval` готує кадр так само, як для попередніх моделей: заливає ignored-зони сірим, зменшує до 1024 по довшій стороні
й запам'ятовує `scale`, щоб повернути бокси в оригінальні координати. Аугментацій немає, бо це оцінка.

In [3]:
#COCO JSON -> DataFrames
def load_coco_as_df(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    imgs = pd.DataFrame(data['images'])
    anns = pd.DataFrame(data['annotations'])
    imgs['video'] = imgs['file_name'].apply(lambda x: x.rsplit('/', 1)[0] if '/' in x else x.rsplit('_', 1)[0])
    imgs['frame_id'] = imgs['file_name'].apply(lambda x: x.rsplit('/', 1)[-1] if '/' in x else x)
    return imgs, anns

# Частка площі об'єкта всередині ignored-зони
def ioa_boxes(objs, zones):
    o = np.asarray(objs, float); z = np.asarray(zones, float)
    iw = np.clip(np.minimum((o[:,0]+o[:,2])[:,None], (z[:,0]+z[:,2])[None]) -
                 np.maximum(o[:,0][:,None], z[:,0][None]), 0, None)
    ih = np.clip(np.minimum((o[:,1]+o[:,3])[:,None], (z[:,1]+z[:,3])[None]) -
                 np.maximum(o[:,1][:,None], z[:,1][None]), 0, None)
    return ((iw*ih) / (o[:,2]*o[:,3])[:,None]).max(axis=1)

# Dataset для оцінки (без аугментацій)
class VisDroneEval(Dataset):
    def __init__(self, split, imgs, frame_step=1):
        self.split = split
        imgs = imgs.sort_values(['video', 'frame_id'])
        imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])
        self.imgs = imgs.reset_index(drop=True)
        self.zones = {k: g[g['iscrowd'] == 1]['bbox'].tolist()
                      for k, g in (train_anns if split == 'train' else val_anns).groupby('image_id')}

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        row = self.imgs.iloc[i]
        img = cv2.cvtColor(cv2.imread(f"{ROOT}/{self.split}/{row['file_name']}"), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        for x, y, w, h in self.zones.get(row['id'], []):           # заливаємо ignored-зони сірим
            img[int(y):int(y+h), int(x):int(x+w)] = GRAY
        s = IMG_SIZE / max(H, W)
        img = cv2.resize(img, (round(W*s), round(H*s)), interpolation=cv2.INTER_AREA)
        t = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0
        return t, {'image_id': int(row['id']), 'scale': s}

def collate(b):
    return tuple(zip(*b))

train_imgs, train_anns = load_coco_as_df(f'{ROOT}/train/annotations_coco.json')
val_imgs,   val_anns   = load_coco_as_df(f'{ROOT}/val/annotations_coco.json')
print('train кадрів:', len(train_imgs), '| val кадрів:', len(val_imgs))

train кадрів: 24201 | val кадрів: 2846


Ultralytics потребує YOLO-формат: окремий `.txt` на кадр, рядок `клас x_c y_c w h` (нормовано 0..1).
Генеруємо його з COCO JSON за тими самими правилами, що й для інших моделей: ignored-зони не є об'єктами,
об'єкти більш ніж на 50% у зоні відкидаються, класи 1..8 → 0..7. Видаляємо старий кеш Ultralytics і створюємо `data.yaml`.
Відмінність: при навчанні RT-DETR ignored-зони на кадрах не зафарбовані сірим, бо для цього потрібно було б перезаписати дуже багато картинок.

In [4]:
# Ultralytics навчається на YOLO-txt розмітці. Генеруємо її з COCO JSON з тією самою логікою, що й для інших:
def write_yolo_labels(split, imgs, anns):
    by_img = {k: g for k, g in anns.groupby('image_id')}
    img_paths = [f'{ROOT}/{split}/{fn}' for fn in imgs['file_name']]
    lbl_paths = img2label_paths(img_paths)
    n_obj = 0
    for (_, row), lp in zip(imgs.iterrows(), lbl_paths):
        lines = []
        g = by_img.get(row['id'])
        if g is not None:
            zones = g[g['iscrowd'] == 1]['bbox'].tolist()
            objs = g[g['iscrowd'] == 0]
            ob, lb = objs['bbox'].tolist(), objs['category_id'].values
            if zones and ob:
                keep = ioa_boxes(ob, zones) <= 0.5
                ob = [b for b, k in zip(ob, keep) if k]; lb = lb[keep]
            W, H = row['width'], row['height']
            for (x, y, w, h), c in zip(ob, lb):
                x1, y1 = max(x, 0), max(y, 0)
                x2, y2 = min(x + w, W), min(y + h, H)
                if x2 - x1 <= 1 or y2 - y1 <= 1:
                    continue
                lines.append(f'{int(c)-1} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
        os.makedirs(os.path.dirname(lp), exist_ok=True)
        with open(lp, 'w') as f:
            f.write('\n'.join(lines))
        n_obj += len(lines)
    print(f'{split}: {len(lbl_paths)} файлів розмітки, {n_obj} об\'єктів')

write_yolo_labels('train', train_imgs, train_anns)
write_yolo_labels('val',   val_imgs,   val_anns)

# прибираємо старі кеші Ultralytics, щоб він перечитав нову розмітку
for c in glob.glob(f'{ROOT}/**/*.cache', recursive=True):
    os.remove(c)

os.makedirs('/content/data', exist_ok=True)
with open('/content/data/data.yaml', 'w') as f:
    f.write(f"path: {ROOT}\ntrain: train\nval: val\n\nnc: 8\nnames: {CLASSES}\n")
print(open('/content/data/data.yaml').read())

train: 24201 файлів розмітки, 1093783 об'єктів
val: 2846 файлів розмітки, 112936 об'єктів
path: /content/dataset
train: train
val: val

nc: 8
names: ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']



Завантажуємо правильні відповіді (GT) для train і val, з якими порівнюватимемо передбачення.
`ap_from_eval` дістає з результатів COCOeval три метрики: mAP (IoU 0.50:0.95), AP50 (IoU 0.5)
та AP_small (об'єкти < 32×32 px). Функція та сама, що для інших моделей, тож метрики порівнювані.

In [5]:
train_gt = COCO(f'{ROOT}/train/annotations_coco.json')
val_gt   = COCO(f'{ROOT}/val/annotations_coco.json')
assert sorted(val_gt.getCatIds()) == list(range(1, 9)), val_gt.getCatIds()

def ap_from_eval(E, iou=None, area='all', cls=None):
    p = E.eval['precision']
    a = {'all': 0, 'small': 1, 'medium': 2, 'large': 3}[area]
    p = p[:, :, :, a, -1]
    if cls is not None:
        p = p[:, :, cls:cls+1]
    if iou is not None:
        t = int(np.where(np.isclose(E.params.iouThrs, iou))[0][0])
        p = p[t:t+1]
    p = p[p > -1]
    return float(p.mean()) if p.size else float('nan')

loading annotations into memory...
Done (t=6.25s)
creating index...
index created!
loading annotations into memory...
Done (t=0.86s)
creating index...
index created!


Спільний протокол для обох варіантів: 3 епохи, imgsz 1024, batch 8, max 300 детекцій, фіксований seed.
`save_period=1` зберігає ваги після кожної епохи на Drive.
Варіанти: **baseline** (налаштування Ultralytics за замовчуванням, `optimizer=auto`) та
**tuned** (AdamW з меншим learning rate `lr0=1e-4`, що спадає до `lr0·0.01`), бо трансформери стабільніше вчаться з меншим LR.
Додаємо аугументацію максимально наюлижену до минулих моделей.


In [6]:
RTDETR_EPOCHS  = 3
RTDETR_IMGSZ   = 1024
RTDETR_PROJECT = f'{CKPT_DIR}/rtdetr_runs'
os.makedirs(RTDETR_PROJECT, exist_ok=True)

RTDETR_COMMON = dict(
    data='/content/data/data.yaml',
    epochs=RTDETR_EPOCHS,
    imgsz=RTDETR_IMGSZ,
    batch=BATCH,
    max_det=300,
    seed=SEED,
    deterministic=True,
    workers=2,
    project=RTDETR_PROJECT,
    exist_ok=True,
    save=True,
    save_period=1,

    fliplr=0.5,
    scale=0.25,
    translate=0.1,
    hsv_h=0.0,
    hsv_s=0.3,
    hsv_v=0.25,
    mosaic=0.0,
    mixup=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
)

RTDETR_VARIANTS = {
    'rtdetr_baseline': {},
    'rtdetr_tuned':    {'optimizer': 'AdamW',
                        'lr0': 1e-4,
                        'lrf': 0.01},
}

def run_dir(name):
    return f'{RTDETR_PROJECT}/{name}'

def training_finished(name):
    last = f'{run_dir(name)}/weights/last.pt'
    if not os.path.exists(last):
        return False
    return torch.load(last, map_location='cpu', weights_only=False).get('epoch', 0) == -1

Донавчаємо RT-DETR-l з попередньо навчених (на COCO) ваг для кожного варіанту.
Якщо варіант уже навчений, він пропускається; якщо навчання обірвалось, воно продовжується з `last.pt` (`resume=True`).
Ваги кожної епохи зберігаються на Drive.

In [ ]:
for name, params in RTDETR_VARIANTS.items():
    last = f'{run_dir(name)}/weights/last.pt'
    print('\n' + '=' * 60 + f'\n RT-DETR: {name} | {params}\n' + '=' * 60)

    if training_finished(name):
        print('Вже навчено, пропускаю')
    elif os.path.exists(last):
        print('Знайдено незавершене навчання -> resume')
        RTDETR(last).train(resume=True)
    else:
        RTDETR('rtdetr-l.pt').train(name=name, **RTDETR_COMMON, **params)

    print('Чекпойнти:', sorted(os.listdir(f'{run_dir(name)}/weights')))
    torch.cuda.empty_cache()


 RT-DETR: rtdetr_baseline | {}
WARNING ⚠️ Download failure, retrying 1/3 in 5s https://github.com/ultralytics/assets/releases/download/v8.4.0/rtdetr-l.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/rtdetr-l.pt
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, h

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/3      12.1G     0.7188     0.4594     0.1137         28        960: 100% ━━━━━━━━━━━━ 3026/3026 1.0it/s 49:28
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.3it/s 2:22
                   all       2846     112936      0.518      0.464      0.434      0.217

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/3      12.4G     0.4895     0.4198    0.04876         38        960: 100% ━━━━━━━━━━━━ 3026/3026 1.0it/s 49:20
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.3it/s 2:18
                   all       2846     112936      0.602      0.432      0.406      0.202

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        3/3      12.4G     0.4578     0.4051    0.04426        299        960: 23% ━━╸───────── 697/3026 1.0it/s 11:22<37:24

Train: 1500 випадкових кадрів з тих, на яких вчилися (кожен 3-й кадр). Фіксований `SEED` гарантує,
що це ті самі 1500 кадрів, що й для інших моделей. Val: повний валідаційний набір.
Порівняння train і val показує, чи модель перенавчається.

In [7]:
N_TRAIN_EVAL = 1500

train_eval_full = VisDroneEval('train', train_imgs, frame_step=FRAME_STEP)
rng = np.random.RandomState(SEED)
train_eval_idx = np.sort(rng.choice(len(train_eval_full), size=N_TRAIN_EVAL, replace=False))
train_eval_dl = DataLoader(Subset(train_eval_full, train_eval_idx.tolist()), batch_size=BATCH,
                           shuffle=False, num_workers=2, collate_fn=collate)

val_eval_ds = VisDroneEval('val', val_imgs, frame_step=1)
val_eval_dl = DataLoader(val_eval_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)

print(f'Train-підмножина: {N_TRAIN_EVAL} кадрів з {len(train_eval_full)} | Val: {len(val_eval_ds)} кадрів')

/tmp/ipykernel_3921/3526284921.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])
/tmp/ipykernel_3921/3526284921.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])


Train-підмножина: 1500 кадрів з 8090 | Val: 2846 кадрів



`epoch_checkpoints` знаходить ваги кожної епохи (1, 2, 3) незалежно від того, як Ultralytics назвав файли.
`evaluate_rtdetr` запускає модель на кадрах (`conf=0.001`, щоб не занизити mAP), повертає рамки в оригінальні
координати, класи 0..7 → 1..8, і рахує через COCOeval mAP, AP50 та AP_small, так само як для інших моделей.
RT-DETR не використовує NMS: завдяки bipartite matching він видає одну рамку на об'єкт.

In [8]:
def epoch_checkpoints(name, n_epochs=RTDETR_EPOCHS):
    wdir = f'{run_dir(name)}/weights'
    files = sorted(glob.glob(f'{wdir}/epoch*.pt'),
                   key=lambda p: int(re.findall(r'epoch(\d+)', os.path.basename(p))[0]))
    if len(files) < n_epochs and os.path.exists(f'{wdir}/last.pt'):
        files.append(f'{wdir}/last.pt')
    return {i + 1: f for i, f in enumerate(files[:n_epochs])}

@torch.no_grad()
def evaluate_rtdetr(weights, dl, gt_coco):
    m = RTDETR(weights)
    results, img_ids = [], []
    for imgs, tgts in dl:
        # тензор RGB [0,1] -> numpy BGR uint8 (формат, який чекає Ultralytics)
        np_imgs = [(t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)[:, :, ::-1].copy()
                   for t in imgs]
        preds = m.predict(np_imgs, imgsz=RTDETR_IMGSZ, conf=0.001, max_det=300,
                          half=True, verbose=False)
        for p, t in zip(preds, tgts):
            s, iid = t['scale'], t['image_id']
            img_ids.append(iid)
            b = p.boxes.xyxy.cpu().numpy() / s
            for (x1, y1, x2, y2), sc, c in zip(b, p.boxes.conf.cpu().numpy(), p.boxes.cls.cpu().numpy()):
                results.append({'image_id': iid, 'category_id': int(c) + 1,
                                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                                'score': float(sc)})
    del m; torch.cuda.empty_cache()
    if not results:
        return {'mAP': 0.0, 'AP50': 0.0, 'AP_small': 0.0}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt_coco.loadRes(results)
        E = COCOeval(gt_coco, dt, 'bbox')
        E.params.imgIds = img_ids
        E.params.maxDets = [1, 10, 300]
        E.evaluate(); E.accumulate()
    return {'mAP':      ap_from_eval(E),
            'AP50':     ap_from_eval(E, iou=0.5),
            'AP_small': ap_from_eval(E, area='small')}


Для кожного варіанту (baseline, tuned), кожної епохи (1–3) і кожного набору (train 1500, val)
рахуємо mAP, AP50, AP_small. Кожен рядок одразу зберігається в CSV на Drive, тож якщо Colab відключиться,
повторний запуск пропустить уже пораховане й продовжить з того місця, де зупинився.

In [9]:
OUT_CSV_LONG_DETR = f'{CKPT_DIR}/rtdetr_per_epoch_metrics.csv'
OUT_CSV_WIDE_DETR = f'{CKPT_DIR}/rtdetr_per_epoch_metrics_wide.csv'

if os.path.exists(OUT_CSV_LONG_DETR):
    df_detr = pd.read_csv(OUT_CSV_LONG_DETR)
    print(f'Знайдено попередні результати: {len(df_detr)} рядків, продовжую')
else:
    df_detr = pd.DataFrame(columns=['Experiment', 'Epoch', 'Split', 'mAP', 'AP50', 'AP_small'])

SPLITS = [('Train (1500 rand)', train_eval_dl, train_gt),
          ('Val (full)',        val_eval_dl,   val_gt)]

for name in RTDETR_VARIANTS:
    ckpts = epoch_checkpoints(name)
    print(f'\n{name}: ' + ', '.join(f'ep{k} -> {os.path.basename(v)}' for k, v in ckpts.items()))
    for ep, ckpt_path in ckpts.items():
        for split_name, dl, gt in SPLITS:
            done = ((df_detr['Experiment'] == name) & (df_detr['Epoch'] == ep) &
                    (df_detr['Split'] == split_name)).any()
            if done:
                continue
            t0 = time.time()
            r = evaluate_rtdetr(ckpt_path, dl, gt)
            row = {'Experiment': name, 'Epoch': ep, 'Split': split_name,
                   'mAP': round(r['mAP'], 4), 'AP50': round(r['AP50'], 4),
                   'AP_small': round(r['AP_small'], 4)}
            df_detr = pd.concat([df_detr, pd.DataFrame([row])], ignore_index=True)
            df_detr.to_csv(OUT_CSV_LONG_DETR, index=False)
            print(f"{name:16s} ep{ep} | {split_name:18s} | mAP={row['mAP']:.4f} "
                  f"AP50={row['AP50']:.4f} AP_small={row['AP_small']:.4f} | {time.time()-t0:.0f}s")

Знайдено попередні результати: 12 рядків, продовжую

rtdetr_baseline: ep1 -> epoch0.pt, ep2 -> epoch1.pt, ep3 -> epoch2.pt

rtdetr_tuned: ep1 -> epoch0.pt, ep2 -> epoch1.pt, ep3 -> epoch2.pt


###Підсумкова таблиця

In [10]:
df_detr['Epoch'] = df_detr['Epoch'].astype(int)
df_detr = df_detr.sort_values(['Experiment', 'Epoch', 'Split']).reset_index(drop=True)

df_detr_wide = df_detr.pivot_table(index=['Experiment', 'Epoch'], columns='Split',
                                   values=['mAP', 'AP50', 'AP_small'])
df_detr_wide.columns = [f'{split} {metric}' for metric, split in df_detr_wide.columns]
order = [f'{s} {m}' for s in ['Train (1500 rand)', 'Val (full)'] for m in ['mAP', 'AP50', 'AP_small']]
df_detr_wide = df_detr_wide[[c for c in order if c in df_detr_wide.columns]].reset_index()

df_detr.to_csv(OUT_CSV_LONG_DETR, index=False)
df_detr_wide.to_csv(OUT_CSV_WIDE_DETR, index=False)

print('=' * 100)
print('RT-DETR: mAP / AP50 / AP_small по епохах (Train = 1500 випадкових кадрів, Val = повний)')
print('=' * 100)
print(df_detr_wide.to_string(index=False))
print('=' * 100)
print(f'Збережено на Drive:\n  {OUT_CSV_LONG_DETR}\n  {OUT_CSV_WIDE_DETR}')
display(df_detr_wide)

RT-DETR: mAP / AP50 / AP_small по епохах (Train = 1500 випадкових кадрів, Val = повний)
     Experiment  Epoch  Train (1500 rand) mAP  Train (1500 rand) AP50  Train (1500 rand) AP_small  Val (full) mAP  Val (full) AP50  Val (full) AP_small
rtdetr_baseline      1                 0.5115                  0.8496                      0.3706          0.2205           0.4357               0.1145
rtdetr_baseline      2                 0.5699                  0.9019                      0.4306          0.1993           0.3977               0.0968
rtdetr_baseline      3                 0.6055                  0.9196                      0.4826          0.1976           0.3907               0.0900
   rtdetr_tuned      1                 0.1116                  0.2217                      0.0519          0.0744           0.1536               0.0350
   rtdetr_tuned      2                 0.4109                  0.7284                      0.2505          0.2067           0.3975               0.0765


,Experiment,Epoch,Train (1500 rand) mAP,Train (1500 rand) AP50,Train (1500 rand) AP_small,Val (full) mAP,Val (full) AP50,Val (full) AP_small
0,rtdetr_baseline,1,0.5115,0.8496,0.3706,0.2205,0.4357,0.1145
1,rtdetr_baseline,2,0.5699,0.9019,0.4306,0.1993,0.3977,0.0968
2,rtdetr_baseline,3,0.6055,0.9196,0.4826,0.1976,0.3907,0.0900
3,rtdetr_tuned,1,0.1116,0.2217,0.0519,0.0744,0.1536,0.0350
4,rtdetr_tuned,2,0.4109,0.7284,0.2505,0.2067,0.3975,0.0765
5,rtdetr_tuned,3,0.4997,0.8285,0.3575,0.2179,0.4145,0.0784


In [11]:
from ultralytics.models.rtdetr.train import RTDETRTrainer

BBLR_NAME = 'rtdetr_bb_lr'
BBLR_PARAMS = {'optimizer': 'AdamW', 'lr0': 1e-4, 'lrf': 0.01}
BACKBONE_LR_MULT = 0.1

RTDETR_VARIANTS[BBLR_NAME] = BBLR_PARAMS


class BackboneLRTrainer(RTDETRTrainer):

    def build_optimizer(self, model, *args, **kwargs):
        opt = super().build_optimizer(model, *args, **kwargs)

        layers = list(getattr(model, 'module', model).model)
        n_bb = 0
        for m in layers:
            if type(m).__name__ in ('HGStem', 'HGBlock', 'DWConv'):
                n_bb += 1
            else:
                break
        bb_ids = {id(p) for m in layers[:n_bb] for p in m.parameters()}

        new_groups = []
        for g in opt.param_groups:
            for is_bb in (False, True):
                ps = [p for p in g['params'] if (id(p) in bb_ids) == is_bb]
                if ps:
                    ng = {k: v for k, v in g.items() if k != 'params'}
                    ng['params'] = ps
                    ng['lr'] = g['lr'] * (BACKBONE_LR_MULT if is_bb else 1.0)
                    ng['part'] = 'backbone' if is_bb else 'transformer'
                    new_groups.append(ng)

        opt.param_groups.clear()
        for ng in new_groups:
            opt.add_param_group(ng)

        print(f'Backbone: шари 0..{n_bb-1}')
        for ng in opt.param_groups:
            print(f"  {ng['part']:11s} | {ng['param_group']:6s} | lr={ng['lr']:.1e} | тензорів: {len(ng['params'])}")
        return opt


last = f'{run_dir(BBLR_NAME)}/weights/last.pt'
print('=' * 60 + f'\n RT-DETR: {BBLR_NAME} | {BBLR_PARAMS} | backbone LR x{BACKBONE_LR_MULT}\n' + '=' * 60)

if training_finished(BBLR_NAME):
    print('Вже навчено, пропускаю')
elif os.path.exists(last):
    print('Знайдено незавершене навчання -> resume')
    RTDETR(last).train(resume=True, trainer=BackboneLRTrainer)
else:
    RTDETR('rtdetr-l.pt').train(name=BBLR_NAME, trainer=BackboneLRTrainer,
                                **RTDETR_COMMON, **BBLR_PARAMS)

print('Чекпойнти:', sorted(os.listdir(f'{run_dir(BBLR_NAME)}/weights')))
torch.cuda.empty_cache()

 RT-DETR: rtdetr_bb_lr | {'optimizer': 'AdamW', 'lr0': 0.0001, 'lrf': 0.01} | backbone LR x0.1
Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.3, hsv_v=0.25, imgsz=960, iou=None, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosai

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/3      12.1G      1.064     0.5718      0.148         28        960: 100% ━━━━━━━━━━━━ 3026/3026 1.0it/s 48:36
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.3it/s 2:20
                   all       2846     112936      0.176      0.265      0.104     0.0325

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/3      12.4G     0.8864     0.5629     0.0972         38        960: 100% ━━━━━━━━━━━━ 3026/3026 1.0it/s 48:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.3it/s 2:16
                   all       2846     112936      0.436      0.402      0.376      0.194

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        3/3      12.4G      0.607     0.5146    0.06797         18        960: 100% ━━━━━━━━━━━━ 3026/3026 1.0it/s 48:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.3it/s 2:22
                   all       2846     112936      0.514      0.408      0.409      0.208

3 epochs completed in 2.558 hours.
Optimizer stripped from /content/drive/MyDrive/visdrone_ckpt/rtdetr_runs/rtdetr_bb_lr/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/visdrone_ckpt/rtdetr_runs/rtdetr_bb_lr/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/visdrone_ckpt/rtdetr_runs/rtdetr_bb_lr/weights/best.pt...
Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 1.2it/s 2:23
           

In [12]:
OUT_CSV_LONG_DETR = f'{CKPT_DIR}/rtdetr_per_epoch_metrics.csv'
OUT_CSV_WIDE_DETR = f'{CKPT_DIR}/rtdetr_per_epoch_metrics_wide.csv'

if os.path.exists(OUT_CSV_LONG_DETR):
    df_detr = pd.read_csv(OUT_CSV_LONG_DETR)
else:
    df_detr = pd.DataFrame(columns=['Experiment', 'Epoch', 'Split', 'mAP', 'AP50', 'AP_small'])

SPLITS = [('Train (1500 rand)', train_eval_dl, train_gt),
          ('Val (full)',        val_eval_dl,   val_gt)]

ckpts = epoch_checkpoints(BBLR_NAME)
print(f'{BBLR_NAME}: ' + ', '.join(f'ep{k} -> {os.path.basename(v)}' for k, v in ckpts.items()))
for ep, ckpt_path in ckpts.items():
    for split_name, dl, gt in SPLITS:
        done = ((df_detr['Experiment'] == BBLR_NAME) & (df_detr['Epoch'] == ep) &
                (df_detr['Split'] == split_name)).any()
        if done:
            continue
        t0 = time.time()
        r = evaluate_rtdetr(ckpt_path, dl, gt)
        row = {'Experiment': BBLR_NAME, 'Epoch': ep, 'Split': split_name,
               'mAP': round(r['mAP'], 4), 'AP50': round(r['AP50'], 4),
               'AP_small': round(r['AP_small'], 4)}
        df_detr = pd.concat([df_detr, pd.DataFrame([row])], ignore_index=True)
        df_detr.to_csv(OUT_CSV_LONG_DETR, index=False)
        print(f"{BBLR_NAME:16s} ep{ep} | {split_name:18s} | mAP={row['mAP']:.4f} "
              f"AP50={row['AP50']:.4f} AP_small={row['AP_small']:.4f} | {time.time()-t0:.0f}s")

df_detr['Epoch'] = df_detr['Epoch'].astype(int)
df_detr = df_detr.sort_values(['Experiment', 'Epoch', 'Split']).reset_index(drop=True)

df_detr_wide = df_detr.pivot_table(index=['Experiment', 'Epoch'], columns='Split',
                                   values=['mAP', 'AP50', 'AP_small'])
df_detr_wide.columns = [f'{split} {metric}' for metric, split in df_detr_wide.columns]
order = [f'{s} {m}' for s in ['Train (1500 rand)', 'Val (full)'] for m in ['mAP', 'AP50', 'AP_small']]
df_detr_wide = df_detr_wide[[c for c in order if c in df_detr_wide.columns]].reset_index()

df_detr.to_csv(OUT_CSV_LONG_DETR, index=False)
df_detr_wide.to_csv(OUT_CSV_WIDE_DETR, index=False)

print('\n' + '=' * 100)
print('RT-DETR: mAP / AP50 / AP_small по епохах (Train = 1500 випадкових кадрів, Val = повний)')
print('=' * 100)
print(df_detr_wide.to_string(index=False))
print('=' * 100)
print(f'Збережено на Drive:\n  {OUT_CSV_LONG_DETR}\n  {OUT_CSV_WIDE_DETR}')
display(df_detr_wide)

print('\nПорівняння: rtdetr_tuned vs rtdetr_bb_lr')
display(df_detr_wide[df_detr_wide['Experiment'].isin(['rtdetr_tuned', BBLR_NAME])])

rtdetr_bb_lr: ep1 -> epoch0.pt, ep2 -> epoch1.pt, ep3 -> epoch2.pt
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is de

,Experiment,Epoch,Train (1500 rand) mAP,Train (1500 rand) AP50,Train (1500 rand) AP_small,Val (full) mAP,Val (full) AP50,Val (full) AP_small
0,rtdetr_baseline,1,0.5115,0.8496,0.3706,0.2205,0.4357,0.1145
1,rtdetr_baseline,2,0.5699,0.9019,0.4306,0.1993,0.3977,0.0968
2,rtdetr_baseline,3,0.6055,0.9196,0.4826,0.1976,0.3907,0.0900
3,rtdetr_bb_lr,1,0.0527,0.1456,0.0193,0.0310,0.0956,0.0080
4,rtdetr_bb_lr,2,0.3464,0.6073,0.2048,0.1825,0.3534,0.0725
5,rtdetr_bb_lr,3,0.4151,0.7174,0.2672,0.2005,0.3933,0.0808
6,rtdetr_tuned,1,0.1116,0.2217,0.0519,0.0744,0.1536,0.0350
7,rtdetr_tuned,2,0.4109,0.7284,0.2505,0.2067,0.3975,0.0765
8,rtdetr_tuned,3,0.4997,0.8285,0.3575,0.2179,0.4145,0.0784



Порівняння: rtdetr_tuned vs rtdetr_bb_lr


,Experiment,Epoch,Train (1500 rand) mAP,Train (1500 rand) AP50,Train (1500 rand) AP_small,Val (full) mAP,Val (full) AP50,Val (full) AP_small
3,rtdetr_bb_lr,1,0.0527,0.1456,0.0193,0.0310,0.0956,0.0080
4,rtdetr_bb_lr,2,0.3464,0.6073,0.2048,0.1825,0.3534,0.0725
5,rtdetr_bb_lr,3,0.4151,0.7174,0.2672,0.2005,0.3933,0.0808
6,rtdetr_tuned,1,0.1116,0.2217,0.0519,0.0744,0.1536,0.0350
7,rtdetr_tuned,2,0.4109,0.7284,0.2505,0.2067,0.3975,0.0765
8,rtdetr_tuned,3,0.4997,0.8285,0.3575,0.2179,0.4145,0.0784
